# Setting up

<U>Google Colab</u>: **work_dir** should point to the location where you uploaded the folder **_Images** to your Google Drive. <br>
Run the code cell below once to mount your Google Drive and authorize access.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
work_dir = '/content/drive/MyDrive/Day1_Notebooks_and_Images-main/_Images/'

<U>Jupyter</u>: work_dir should point to the local storage location where you uploaded the folder **_Images**. <br>
Run the code cell below once to install required dependencies and set **work_dir**.

In [ ]:
!pip install numpy matplotlib scikit-image --quiet
work_dir = 'D:/_Images/'

# 1. Loading and Displaying Images

The Python package [Scikit-Image](https://scikit-image.org) gathers many image processing functions organized as a folder of code scripts.</br>To <u>import</u> a function from a Python package, it must first be <u>installed</u> in your environment (Google Colab comes with a lot of preinstalled packages). Use the statement `from` to select the package where the function is defined and `import` to import it:

In [ ]:
from skimage.io import imread

Large packages are often organised as a hierarchy of <u>subpackages</u> (here `io` inside `skimage`), accessed by the <u>dot operator</u>. </br>If you do not remember the exact name of a package or function, start typing the first letters after `from` or `import` to get suggestions.</br> Package versions are extremely important for reproducibility, especially when they have dependencies. <br>In Colab notebooks you often get very recent versions of the pre-installed packages:</br>

In [ ]:
import skimage
print(skimage.__version__)

Now that we imported the function **imread**, it's time to <u>call</u> it! <br>
The only <u>mandatory argument</u> is a string containing the <u>path</u> to the image file to load. <br>
The function <u>returns</u> the content of the image and we store it in the variable _img_:

In [ ]:
img = imread(work_dir+'Blobs.tif')

To print the <u>type</u>, <u>shape</u> (dimensions) and <u>data type</u> of the variable _img_, use:

In [ ]:
print(type(img), img.shape, img.dtype)

<u>Numpy arrays</u> are some sort of <u>multidimensional lists</u> (multiple dimensions, each with associated index). <br>
Here `img` is a 2 dimensional array of size 254 rows x 256 columns, and each element is a 8-bit unsigned integer.

![](https://www.w3resource.com/w3r_images/numpy-1d2d3d-array.png)

Next, we import `pyplot` from the `matplotlib` package to display the image.</br>
Programmers being lazy, they often use the `as` statement to import functions with a shorter name to limit further typing:

In [ ]:
from matplotlib import pyplot as plt
plt.imshow(img, cmap='grey')
plt.show()

`Pyplot` submodule is a <u>collection of plotting functions</u> accessed with the <u>dot operator</u>. <br>
For instance, the function `imshow` is used to display an image by passing the intensity values of its pixels (here as a Numpy array).

<font color="green"><font><u>Exercise</u>:</font><br>
- Explore the different functions available in `Pyplot` by typing `plt.` and  pressing *Tab* for completions<br>
- Find a function to add a title to the image<br>
- Now hover over `imshow` and explore its arguments and associated variable type <br>
- Try to understand the `vmin`, `vmax` arguments by experimenting </br>
- Finally, set the `cmap` argument to the strings 'rainbow' or 'viridis' and interpret the effect<br><br>
You can get more information on Colormaps from [Matplotlib documentation](https://matplotlib.org/stable/users/explain/colors/colormaps.html).

# 2. Multidimensional Images, Contrast Adjustement and Cropping



Let's load another image:

In [ ]:
img = imread(work_dir+'Cells.tif')
print(type(img), img.shape, img.dtype)

and visualize it as before:

In [ ]:
plt.imshow(img)
plt.show()

`imshow` interprets the third dimension as a color channel. Here this is not a bad guess for visualization purpose but it is actually a three channel fluorescence image (not a color camera image!). For more than three channels, the display would be meaningless!

Gamma correction is a convenient technique to quickly adjust the display contrast of an image:

In [ ]:
from skimage.exposure import adjust_gamma

plt.imshow(adjust_gamma(img, 0.6))
plt.show()

It is possible to extract a specific channel by setting the third dimension index. <br>
We are then left with a 2D array that can be 'safely' displayed:

In [ ]:
plt.imshow(img[:, :, 2], cmap='grey', vmax=75)
plt.show()

Note how we used colon `:` to specify that we want to keep all the elements from the first and second dimension.<br>

<font color="green"><font><u>Exercise</u>: Create a Code Cell below and use list slicing to extract the region from rows 200 to 400 and columns 350 to 550 of the third channel, store it to a new image `roi` and visualize it with a gamma correction of 0.5.</font>

## Solution

In [ ]:
roi = img[200:400, 350:550, 2]
plt.imshow(adjust_gamma(roi, 0.5), cmap='grey')
plt.show()

## Extra: Subplotting

In [ ]:
# Create a 1 row x 3 columns figure montage
fig, ax = plt.subplots(1, 3, figsize=(12, 12))

roi = img[200:401, 350:551, :]

# Plot the different channels to the figure
for i in range(3):
  ax[i].imshow(roi[:, :, i], cmap ='gray', vmax=75)
  ax[i].set_axis_off()

plt.show()

# 3. Intensity Histograms

To get a better idea of the content of an image, it is useful to plot its <u>intensity histogram</u> with `pyplot.hist`. </br>This function takes a one dimensional Numpy array as input so we first call the `ravel` _method_ to serialize the image:


In [ ]:
img = imread(work_dir+'Blobs.tif')
pix_values = img.ravel()
print(type(pix_values), pix_values.shape)

Note that the serialization starts from last to first axis so it is row by row for a 2D image (C-style).

To plot the histogram with 64 bins:

In [ ]:
h = plt.hist(pix_values, bins=64)
plt.show()

<font color="green"><font><u>Exercise</u>: How many intensity populations do you observe? Why do you think there are gaps in the histogram?<br>
Specifically, what transformation was probably applied to this image? Run and interpret the result of the code below.</font>

In [ ]:
from skimage.filters import gaussian

img_flt = gaussian(img, sigma=1.5, preserve_range=True)
pix_values = img_flt.ravel()
plt.hist(pix_values, bins=64)
plt.show()

# 4. Application: Analyzing Fibers Orientation

We will now build a script to estimate the orientation of colagen fibers imaged by second harmonic microscopy:

In [ ]:
from matplotlib import pyplot as plt

img = imread(work_dir+'Fibers_pref.tif')
plt.imshow(img, cmap='grey')
plt.show()

First, we will compute the <u>intensity gradient</u> at every pixel, that is a vector pointing in the direction of fastest intensity variation. <br>

At a fiber location the direction of the local gradient is <u>orthogonal to the fiber</u>. <br>

The code below filters the image to reduce noise, computes the intensity gradient and estimates the orthogonal direction at each pixel:


In [ ]:
import numpy as np
from skimage.filters import gaussian

img_flt = gaussian(img, sigma=1.5, preserve_range=True)
grad_y, grad_x = np.gradient(img_flt)
img_angle_fibers = np.arctan2(grad_x, grad_y)

Notice how the function `gradient` returns two arrays, each with one component of the <u>gradient vector</u> (same size as `img`).

In [ ]:
plt.imshow(img_angle_fibers, cmap='rainbow')
plt.show()

The angles reported are only meaningful for significant gradient magnitudes, that is in the vicinity of a fiber. <br> In background regions the angle reported is somehow randomly distributed (baseline count in histogram).

In [ ]:
angle_fibers = np.ravel(np.rad2deg(img_angle_fibers))
hst = plt.hist(angle_fibers, bins=45)
plt.show()

Notice how the angles are reported from -180º to 180º, which is a bit meaningless for a fiber orientation (from 0 to 180º).

<font color="green"><font><u>Exercise</u>: By using an operator we studied before, find a way to wrap the angles so that 181º -> 1º, 182º -> 2º ...</font>

The function `hst` returns two arrays: the first one holds the bin counts and the second one holds the bin edges.

In [ ]:
bin_counts = hst[0]
print(bin_counts)

The bin edges alternate bin starts/ends, we hence compute bin centers as their averages:

In [ ]:
bin_edges = hst[1]
bin_centers = (bin_edges[:-1]+bin_edges[1:])/2
print(bin_centers)

Next, we define an intensity offseted (_offs_) Gaussian function (_mu_, _sigma_) with height (_h_):

In [ ]:
def fit_func(x, mu, sigma, h, offs):
    return h*np.exp(-(x-mu)**2/(2*sigma**2))+offs

And we fit it to the counts of the histogram at the bin centers:

In [ ]:
from scipy.optimize import curve_fit

popt, pcov = curve_fit(fit_func, xdata=bin_centers, ydata=bin_counts)
plt.hist(angle_fibers%180, bins=45)
plt.plot(bin_centers, fit_func(bin_centers, *popt), color='red', linewidth=2.5)
plt.show()

Finally, we recover the first two parameters of the fitted function (_mu_ and _sigma_) and we print them:

In [ ]:
print('Mean Orientation: '+str(popt[0]))
print('Orientation spread: '+str(abs(popt[1])))

For a more accurate results, it is here preferable to restrict the fit to a given range:

In [ ]:
mask = bin_centers < 125
popt, pcov = curve_fit(fit_func, xdata=bin_centers[mask], ydata=bin_counts[mask])

plt.hist(angle_fibers%180, bins=45)
plt.plot(bin_centers, fit_func(bin_centers, *popt), color='red', linewidth=2.5)
plt.show()

In [ ]:
print('Mean Orientation: '+str(popt[0]))
print('Orientation spread: '+str(abs(popt[1])))